In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image

# ========= SETTINGS =========
TARGET_SIZE = (224, 224)

# NOTE: ../../ because notebook is inside notebooks/02_preprocess/
RAW_ROOT = Path("../../data/raw/archive (5)")
OUT_ROOT = Path("../../data/processed/archive5")

TRAIN_IMG_DIR = RAW_ROOT / "train" / "train" / "images"
TEST_IMG_DIR  = RAW_ROOT / "test"  / "test"  / "images"

TRAIN_CSV_IN = RAW_ROOT / "train" / "train" / "train.csv"
TEST_CSV_IN  = RAW_ROOT / "test"  / "test"  / "test.csv"

OUT_TRAIN_IMG_DIR = OUT_ROOT / "train" / "images"
OUT_TEST_IMG_DIR  = OUT_ROOT / "test"  / "images"

OUT_TRAIN_CSV = OUT_ROOT / "train.csv"
OUT_TEST_CSV  = OUT_ROOT / "test.csv"

# ========= CHECKS =========
print("RAW_ROOT exists:", RAW_ROOT.exists())
print("TRAIN_CSV exists:", TRAIN_CSV_IN.exists())
print("TEST_CSV exists :", TEST_CSV_IN.exists())
print("TRAIN_IMG_DIR exists:", TRAIN_IMG_DIR.exists())
print("TEST_IMG_DIR exists :", TEST_IMG_DIR.exists())

OUT_TRAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_TEST_IMG_DIR.mkdir(parents=True, exist_ok=True)

# ========= LOAD CSVs =========
train_df = pd.read_csv(TRAIN_CSV_IN)
test_df  = pd.read_csv(TEST_CSV_IN)

print("\nTrain CSV columns:", list(train_df.columns))
print("Test  CSV columns:", list(test_df.columns))

# ========= IMAGE PROCESS FUNCTION =========
def process_one_image(src_path: Path, dst_path: Path):
    with Image.open(src_path) as im:
        im = im.convert("RGB")
        im = im.resize(TARGET_SIZE)
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        im.save(dst_path, format="JPEG", quality=95)

# ========= PROCESS TRAIN =========
train_processed = 0
train_skipped = 0

for _, row in train_df.iterrows():
    fname = row["filename"]
    src = TRAIN_IMG_DIR / fname
    dst = OUT_TRAIN_IMG_DIR / fname

    if not src.exists():
        train_skipped += 1
        continue

    process_one_image(src, dst)
    train_processed += 1

print("\nTRAIN done ", train_processed, "| skipped:", train_skipped)

# ========= PROCESS TEST =========
test_processed = 0
test_skipped = 0

for _, row in test_df.iterrows():
    fname = row["filename"]
    src = TEST_IMG_DIR / fname
    dst = OUT_TEST_IMG_DIR / fname

    if not src.exists():
        test_skipped += 1
        continue

    process_one_image(src, dst)
    test_processed += 1

print("\nTEST done ", test_processed, "| skipped:", test_skipped)

# ========= SAVE CSVs =========
train_df.to_csv(OUT_TRAIN_CSV, index=False)
test_df.to_csv(OUT_TEST_CSV, index=False)

print("\nSaved to:", OUT_ROOT.resolve())


RAW_ROOT exists: True
TRAIN_CSV exists: True
TEST_CSV exists : True
TRAIN_IMG_DIR exists: True
TEST_IMG_DIR exists : True

Train CSV columns: ['image_id', 'filename', 'label']
Test  CSV columns: ['image_id', 'filename']

TRAIN done  7200 | skipped: 0

TEST done  4800 | skipped: 0

Saved to: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5


In [1]:
from pathlib import Path
import pandas as pd

PROC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5")

TRAIN_CSV = PROC / "train.csv"
TEST_CSV  = PROC / "test.csv"

TRAIN_IMG_DIR = PROC / "train" / "images"
TEST_IMG_DIR  = PROC / "test" / "images"

print("VERIFY:", PROC)
print("Train CSV exists:", TRAIN_CSV.exists())
print("Test  CSV exists:", TEST_CSV.exists())
print("Train images dir exists:", TRAIN_IMG_DIR.exists())
print("Test  images dir exists:", TEST_IMG_DIR.exists())

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("\nTrain rows:", len(train_df), "| columns:", list(train_df.columns))
print("Test  rows:", len(test_df),  "| columns:", list(test_df.columns))

# Check image files exist
missing_train = (~train_df["filename"].apply(lambda x: (TRAIN_IMG_DIR / str(x)).exists())).sum()
missing_test  = (~test_df["filename"].apply(lambda x: (TEST_IMG_DIR / str(x)).exists())).sum()

print("\nMissing TRAIN images:", missing_train)
print("Missing TEST images :", missing_test)

# Label distribution (train only)
if "label" in train_df.columns:
    print("\nTrain label counts:")
    print(train_df["label"].value_counts().sort_index())
else:
    print("\nNo label column found in TRAIN (unexpected).")


VERIFY: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5
Train CSV exists: True
Test  CSV exists: True
Train images dir exists: True
Test  images dir exists: True

Train rows: 7200 | columns: ['image_id', 'filename', 'label']
Test  rows: 4800 | columns: ['image_id', 'filename']

Missing TRAIN images: 0
Missing TEST images : 0

Train label counts:
label
1     171
2    2349
3     534
4    2079
5    1185
6     882
Name: count, dtype: int64


In [3]:
!pip install scikit-learn
import shutil
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

PROC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5")
OUT = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5_yolo")

train_csv = PROC / "train.csv"
train_img_dir = PROC / "train" / "images"

df = pd.read_csv(train_csv)
df["label"] = df["label"].astype(str)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train split:", len(train_df))
print("Val split:", len(val_df))

def build_split(split_df, split_name):
    copied = 0
    missing = 0

    for _, row in split_df.iterrows():
        label = row["label"]
        fname = row["filename"]

        src = train_img_dir / fname
        dst_dir = OUT / split_name / label
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / fname

        if src.exists():
            shutil.copy2(src, dst)
            copied += 1
        else:
            missing += 1

    print(f"{split_name}: copied={copied}, missing={missing}")

build_split(train_df, "train")
build_split(val_df, "val")

print("archive5_yolo created at:", OUT)

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Train split: 5760
Val split: 1440
train: copied=5760, missing=0
val: copied=1440, missing=0
archive5_yolo created at: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive5_yolo
